# B2 · Building a Dataset with AI

*Path B: AI-Assisted, notebook 2 of 5 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Prompt for code that reduces records to one row per person
- Merge tables with AI help and check the result
- Test logic on cases where you already know the answer
- Build the variables needed for analysis

**Before this notebook:** [B1: Your First Prompts](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B1_First_Prompts.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Where we are

The demographics table is clean. Now you need one table with one row per person, containing each person's first diagnosis date and first prescription date. (A diagnosis here means dementia or mild cognitive impairment: "the condition".) In this notebook the prompts are yours to write; each task gives you a checklist instead of a template.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal. It repeats the recoding from notebook B1, so that you start from clean data.

> **If you get stuck on a task,** the Solutions section at the end of the notebook has working code for each one. Paste it in and carry on; later tasks depend on earlier ones.

> **Your data are already loaded.** If the AI's answer includes lines that create example data, read a file, or install a package, leave those lines out. Otherwise they may overwrite your tables.

In [ ]:
library(tidyverse)

person_df    <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")
drug_df      <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv")

# Catch-up: the recoding from notebook B1
person_df <- person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other",
      .default = race
    ),
    sex_at_birth = case_when(
      sex_at_birth %in% c("Intersex", "Prefer not to answer", "Skip") ~ "Other",
      .default = sex_at_birth
    )
  )

# Shuffle the records. Real data do not arrive sorted by date.
set.seed(2026)
condition_df <- condition_df %>% slice_sample(prop = 1)
drug_df      <- drug_df %>% slice_sample(prop = 1)

# ---- Helper code. You do not need to read this part; just run the cell. ----
describe_columns <- function(data) {
  tibble(column = names(data), type = map_chr(data, ~ class(.x)[1]))
}
check <- function(ok, label) {
  ok <- tryCatch(suppressWarnings(isTRUE(ok)), error = function(e) FALSE)
  cat(if (ok) "PASS    " else "NOT YET ", label, "\n")
}

## 2. Task 1: one row per person

At the end of the Foundations you saw that `condition_df` has several records for some people, and that joining it directly to `person_df` multiplies the rows. The fix is to keep only each person's **earliest** record.

### Know the answer for one person first

Before asking for code, look at one person by hand. Then you will have a case to check the code against.

In [ ]:
condition_df %>% filter(person_id == 8)

Person 8 has four records, not in date order. The earliest is dated 2025-06-09. (The data are simulated, so the dates mean nothing.) Whatever code you get, **it must keep that row for person 8.**

Use this case to check the code yourself. **Do not put it in your prompt**: with real data, one person's ID and dates are participant-level data.

### The goal

Create `earliest_condition`, containing only the row with each person's earliest `condition_start_date`, and `earliest_drug`, containing only the row with each person's earliest `drug_exposure_start_date`. Each must have exactly one row per person.

### Write your prompt

Get the column descriptions to paste into your prompt:

In [ ]:
describe_columns(condition_df)
describe_columns(drug_df)

Prompt checklist:

- R and tidyverse
- The names of both dataframes
- Their columns and types
- What one row represents, and that a person can have several rows
- The goal: for each person, keep the whole row that has the earliest date, with the original column names
- What to do if a person has two records on the same earliest date (keep only one)
- The names for the two results
- That the dataframes already exist, so no example data is needed

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# Paste the AI's code here, read it, then run it

### Read it

The code almost certainly uses a function you have not met, such as `group_by()`, `slice_min()`, `arrange()`, or `summarise()`. Ask the AI: *"Explain this code line by line for a beginner."* Do not go on until you could explain it to a classmate.

### Check it yourself

Two checks. Is person 8's row the one you found by hand? Does each result have as many rows as it has different people?

In [ ]:
# Your own check

In [ ]:
# Provided check (you do not need to read this code)
check(all(c("person_id", "condition_concept_id", "condition_start_date") %in% names(earliest_condition)),
      "earliest_condition still has its original columns")
check(all(c("person_id", "drug_concept_id", "drug_exposure_start_date") %in% names(earliest_drug)),
      "earliest_drug still has its original columns")
check(nrow(earliest_condition) == 30 && n_distinct(earliest_condition$person_id) == 30,
      "earliest_condition has one row for each of the 30 people with a diagnosis")
check(nrow(earliest_drug) == 35 && n_distinct(earliest_drug$person_id) == 35,
      "earliest_drug has one row for each of the 35 people with a prescription")
check(earliest_condition$condition_start_date[earliest_condition$person_id == 8] == as.Date("2025-06-09"),
      "person 8 has the earliest date, 2025-06-09")
check(nrow(earliest_condition) == 30 && all(earliest_condition$condition_start_date ==
            tapply(condition_df$condition_start_date, condition_df$person_id, min)[as.character(earliest_condition$person_id)]),
      "every person's diagnosis date is their earliest")
check(nrow(earliest_drug) == 35 && all(earliest_drug$drug_exposure_start_date ==
            tapply(drug_df$drug_exposure_start_date, drug_df$person_id, min)[as.character(earliest_drug$person_id)]),
      "every person's prescription date is their earliest")

## 3. Task 2: merge the three tables

### The goal

Create `combined_df` by adding the columns of `earliest_condition` and `earliest_drug` to `person_df`, matching on `person_id`. **All 100 people must be kept**, including those with no diagnosis and no prescription.

From notebook F6, you know which kind of join keeps every row of the first table. Name it in your prompt. If you leave the choice to the AI, it may pick a different join, and people will silently disappear.

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# Paste the AI's code here, read it, then run it

### Check it yourself

A join can drop people or duplicate them. Which two numbers would show that neither happened?

In [ ]:
# Your own check

In [ ]:
# Provided check (you do not need to read this code)
check(nrow(combined_df) == 100, "combined_df has 100 rows")
check(n_distinct(combined_df$person_id) == 100, "each person appears exactly once")
check(sum(!is.na(combined_df$condition_start_date)) == 30, "30 people have a diagnosis date")
check(sum(!is.na(combined_df$drug_exposure_start_date)) == 35, "35 people have a prescription date")

The other people have `NA` in those date columns. Here, `NA` means "no record": the person was never diagnosed, or never prescribed the drug.

## 4. Task 3: who was exposed before diagnosis?

A drug cannot have contributed to a disease if the person started taking it **after** the disease was diagnosed. So a person counts as **exposed** only if they started the drug before their diagnosis, or took it and were never diagnosed.

| Situation | Status |
|---|---|
| No prescription record | "Unexposed" |
| Took the drug, never diagnosed | "Exposed" |
| Started the drug **before** the diagnosis date | "Exposed" |
| Started the drug **on or after** the diagnosis date | "Unexposed (Post-Diagnosis)" |

### Make a test first

Logic like this is easy to get subtly wrong, and the AI can get it wrong too. Before using any code on the real table, try it on a few made-up patients for whom the answer is obvious.

In [ ]:
timing_test <- tibble(
  person_id                = 1:6,
  drug_exposure_start_date = as.Date(c(NA, "2010-01-01", "2010-01-01", "2018-01-01", NA, "2015-01-01")),
  condition_start_date     = as.Date(c(NA, NA, "2015-01-01", "2015-01-01", "2015-01-01", "2015-01-01"))
)
timing_test

(`tibble()` is the tidyverse's version of `data.frame()`.)

**Using the table of rules, decide the status of each of these six patients now, before you have any code.** Write your answers here.

**My predictions:** *(double-click to edit)* 1: ... 2: ... 3: ... 4: ... 5: ... 6: ...

### The goal

Add a column `true_exposure_status` to **both** `timing_test` and `combined_df`, following the four rules in the table.

Prompt checklist: the dataframe names, the two date columns and their type (Date), the fact that either date can be missing, the four rules with the exact labels, and the name of the new column. You can paste the table of rules. Say that both dataframes already exist; otherwise the AI may write code that rebuilds `timing_test`.

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# Paste the AI's code here, read it, then run it

### Check it yourself

Look at `timing_test`. Does every patient have the status you predicted?

In [ ]:
# Your own check

In [ ]:
# Provided check (you do not need to read this code)
check(identical(timing_test$true_exposure_status == "Exposed", c(FALSE, TRUE, TRUE, FALSE, FALSE, FALSE)),
      "test patients 2 and 3 are Exposed; 1, 4, 5, and 6 are not")
check(identical(as.character(timing_test$true_exposure_status),
                c("Unexposed", "Exposed", "Exposed", "Unexposed (Post-Diagnosis)", "Unexposed", "Unexposed (Post-Diagnosis)")),
      "all six test patients have exactly the labels from the table of rules")
check(!anyNA(combined_df$true_exposure_status), "everyone in combined_df has a status")
check(sum(combined_df$true_exposure_status == "Exposed") == 35, "35 people in combined_df are Exposed")

### Why the test table mattered

Run this:

In [ ]:
combined_df %>% count(true_exposure_status)

(If this gives an error, finish Task 3 first.)

Nobody in this dataset started the drug after their diagnosis. So if the code had **ignored the dates completely** and counted everyone with a prescription as exposed, the count for `combined_df` would have been exactly the same, and that check would still have passed. Only test patients 4 and 6 can reveal that mistake.

> **A check can only catch the mistakes your data are able to show.** When the logic matters, build a small test with cases you know.

## 5. Task 4: variables for analysis

### The goal

Add three columns to `combined_df`:

- `age`: 2026 minus `year_of_birth`.
- `condition_status`: the number 1 if the person has a diagnosis (`condition_concept_id` is not missing), the number 0 otherwise.
- `exposure_status`: the number 1 if `true_exposure_status` is "Exposed", the number 0 otherwise.

You can ask for all three in one prompt. Say that the two status columns should be numeric: statistical models need numbers, not the text "1" and "0".

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# Paste the AI's code here, read it, then run it

### Check it yourself

From the earlier tasks you already know how many people have a diagnosis and how many are exposed. Do the new columns agree?

In [ ]:
# Your own check

In [ ]:
# Provided check (you do not need to read this code)
check(abs(mean(combined_df$age) - 52.91) < 0.01, "mean age is 52.91")
check(is.numeric(combined_df$condition_status) && is.numeric(combined_df$exposure_status), "both status columns are numeric")
check(sum(combined_df$condition_status) == 30 && all(combined_df$condition_status %in% c(0, 1)), "condition_status is 1 for 30 people and 0 for the rest")
check(sum(combined_df$exposure_status) == 35 && all(combined_df$exposure_status %in% c(0, 1)), "exposure_status is 1 for 35 people and 0 for the rest")
check(nrow(combined_df) == 100, "combined_df still has 100 rows")

## 6. Task 5: save your work

Everything you have built exists only in R's memory, which Colab wipes when the session ends.

### The goal

Save `combined_df` as a CSV file called `cleaned_dataset.csv`. You have not learned how to do this, so ask the AI. This one needs only a very short prompt.

In [ ]:
# Paste the AI's code here, read it, then run it

In [ ]:
# Provided check (you do not need to read this code)
check(file.exists("cleaned_dataset.csv"), "the file cleaned_dataset.csv exists")

In Colab, the file appears in the **Files** panel (the folder icon in the left sidebar). It is deleted when the session ends, so download it from that panel if you want to keep it.

---
## Recap

| Habit | Why |
|---|---|
| Work out one case by hand first | You have something to check the code against |
| Name the join you want | The AI may otherwise choose one that drops people |
| Compare rows with different people after every step | Catches dropped and duplicated people |
| Build a small test table for any rule-based logic | Your real data may not contain the cases that expose a mistake |
| Check new columns against numbers you already know | Catches logic that is the wrong way round |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Task 1: one row per person**

`group_by(person_id)` means "do the next step separately for each person". `slice_min()` keeps the row with the smallest (earliest) date. `with_ties = FALSE` keeps one row if two share that date. Your AI may instead sort the rows and keep the first one per person; that is also correct.

```r
earliest_condition <- condition_df %>%
  group_by(person_id) %>%
  slice_min(condition_start_date, n = 1, with_ties = FALSE) %>%
  ungroup()

earliest_drug <- drug_df %>%
  group_by(person_id) %>%
  slice_min(drug_exposure_start_date, n = 1, with_ties = FALSE) %>%
  ungroup()
```

**Task 2: merge the three tables**

A left join keeps every row of the first table.

```r
combined_df <- person_df %>%
  left_join(earliest_condition, by = "person_id") %>%
  left_join(earliest_drug, by = "person_id")
```

**Task 3: who was exposed before diagnosis?**

Test patients: 1 Unexposed, 2 Exposed, 3 Exposed, 4 Unexposed (Post-Diagnosis), 5 Unexposed, 6 Unexposed (Post-Diagnosis). The rules about missing dates come first, because comparing a date with a missing date gives `NA`.

```r
timing_test <- timing_test %>%
  mutate(
    true_exposure_status = case_when(
      is.na(drug_exposure_start_date) ~ "Unexposed",
      !is.na(drug_exposure_start_date) & is.na(condition_start_date) ~ "Exposed",
      drug_exposure_start_date < condition_start_date ~ "Exposed",
      drug_exposure_start_date >= condition_start_date ~ "Unexposed (Post-Diagnosis)"
    )
  )

combined_df <- combined_df %>%
  mutate(
    true_exposure_status = case_when(
      is.na(drug_exposure_start_date) ~ "Unexposed",
      !is.na(drug_exposure_start_date) & is.na(condition_start_date) ~ "Exposed",
      drug_exposure_start_date < condition_start_date ~ "Exposed",
      drug_exposure_start_date >= condition_start_date ~ "Unexposed (Post-Diagnosis)"
    )
  )
```

**Task 4: variables for analysis**

```r
combined_df <- combined_df %>%
  mutate(
    age = 2026 - year_of_birth,
    condition_status = if_else(is.na(condition_concept_id), 0, 1),
    exposure_status = if_else(true_exposure_status == "Exposed", 1, 0)
  )
```

**Task 5: save your work**

```r
write_csv(combined_df, "cleaned_dataset.csv")
```

---
## Next

Continue with [B3: Exploring and Plotting with AI](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B3_Exploring_and_Plotting_with_AI.ipynb).